# 3 · The search

The small space could be enumerated: 3234 simulations. A realistic one cannot.
The question of this notebook is how close a **budget** of simulations gets to
the optimum — and, because we have the oracle, we can actually measure it.

## Bayesian optimisation in one paragraph

Spend a few simulations on a spread-out initial sample. Fit a **surrogate** to
what you have seen: a model that, for any configuration, predicts a cost µ and
how unsure it is, σ (in the literature these are often Gaussian processes;
here it is an ensemble of small networks). Pick the next configuration by an
**acquisition** rule that trades a low µ against a high σ — exploit what looks
good, explore what is unknown. Simulate it, refit, repeat until the budget is
spent.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dse
import dse.plots as plots
from dse import DATA, WORK

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

In [ ]:
mlir = DATA / "small" / "gemv.mlir"
oracle = dse.load_pool(DATA / "small" / "pool.csv")
best = oracle["cost"].min()

# Random search needs no simulator once the oracle exists; 2000 trials of it
# give the reference every search below is measured against.
baseline = dse.random_search_baseline(oracle, budget=100)

## Forty simulations

In [ ]:
r40 = dse.bo(mlir, "bo40", max_evals=40, n_init=10)
pool40 = r40.pool()
v = dse.visited(pool40)

print(f"{len(v)} simulations in {r40.seconds:.1f} s")
print(f"best found {v['cost'].min():.3f} ms — rank {dse.rank_of(v['cost'].min(), oracle)} of {len(oracle)}")

Every row the search visited, in order. The first ten are the initial sample —
no surrogate yet. From then on each row carries what the surrogate believed
about it when it was chosen: its prediction `mu`, its uncertainty `sigma`, and
the acquisition value `acq` that made it the pick.

In [ ]:
cols = dse.param_columns(pool40) + ["cost", "mu", "sigma", "acq"]
v[cols].style.format({"cost": "{:.2f}", "mu": "{:.2f}", "sigma": "{:.2f}", "acq": "{:.2f}"}).hide(axis="index")

In [ ]:
plots.surrogate_check(pool40)
plt.show()

Points on the dotted line were predicted right; error bars show how sure the
surrogate was. Early picks are far off with wide bars — it was exploring. The
later ones cluster near the line, low: it had learned where the fast region is.

In [ ]:
ax = plots.best_so_far({"Bayesian, 40 evals": pool40}, oracle_best=best,
                       baseline=baseline[:, :40], n_init=10)
plt.show()

## Is that luck?

One run tells you one thing. Run the same search from five random seeds.

In [ ]:
r5 = dse.bo(mlir, "bo40x5", max_evals=40, n_init=10, seeds=5)
pools5 = r5.pools()

print(f"5 × 40 simulations in {r5.seconds:.0f} s")
for seed, p in sorted(pools5.items()):
    m = dse.visited(p)["cost"].min()
    print(f"seed {seed:4d}: best {m:6.3f} ms  rank {dse.rank_of(m, oracle):4d}  ({(m / best - 1) * 100:4.0f} % above optimum)")

In [ ]:
ax = plots.seeds_spread(pools5, oracle_best=best, baseline=baseline[:, :40])
plt.show()

With a target that is 0.3 % of the space, forty simulations is a coin flip:
some seeds find the fast line, some never leave the bulk. That spread *is* a
result — a single run of a stochastic search is an anecdote.

## More budget

In [ ]:
r100 = dse.bo(mlir, "bo100", max_evals=100, n_init=20)
pool100 = r100.pool()
m = dse.visited(pool100)["cost"].min()
print(f"100 simulations in {r100.seconds:.1f} s — best {m:.3f} ms, rank {dse.rank_of(m, oracle)}")

ax = plots.best_so_far({"Bayesian, 40 evals": pool40, "Bayesian, 100 evals": pool100},
                       oracle_best=best, baseline=baseline, n_init=20)
plt.show()

A hundred simulations is 3 % of the space. Over the seeds we tried beforehand,
four of five land exactly on the optimum with that budget; the fifth never
leaves the bulk. Random search with the same budget is still four to five
times off in the median.

## A space that cannot be enumerated live

4096 × 4096 on up to 2048 DPUs (32 ranks). The number of DPUs is now a
parameter too. We enumerated this space *once*, on 14 cores, in seven minutes,
so that the search can be scored; on one core it would take an hour and a half.

In [ ]:
big_mlir = DATA / "big" / "gemv.mlir"
big_oracle = dse.load_pool(DATA / "big" / "pool.csv")
big_space = dse.load_space(DATA / "big" / "space.json")
big_best = big_oracle["cost"].min()

print(dse.cinm.describe_space(big_space))
print()
print(f"best {big_best:.2f} ms, median {big_oracle['cost'].median():.1f} ms; "
      f"within 10 % of the best: {(big_oracle['cost'] <= 1.1 * big_best).sum()} configurations")

In [ ]:
rb = dse.bo(big_mlir, "big_bo", max_evals=100, n_init=20)
poolb = rb.pool()
mb = dse.visited(poolb)["cost"].min()
big_baseline = dse.random_search_baseline(big_oracle, budget=100)

print(f"100 simulations in {rb.seconds:.1f} s")
print(f"best found {mb:.2f} ms — rank {dse.rank_of(mb, big_oracle)} of {len(big_oracle)}, "
      f"{(mb / big_best - 1) * 100:.0f} % above the optimum")
print(f"random search, same budget: median {np.median(big_baseline[:, -1]):.1f} ms "
      f"({(np.median(big_baseline[:, -1]) / big_best - 1) * 100:.0f} % above)")

In [ ]:
ax = plots.best_so_far({"Bayesian, 100 evals": poolb}, oracle_best=big_best,
                       baseline=big_baseline, n_init=20,
                       title="26,238 feasible configurations — best latency found vs. simulations")
plt.show()

In [ ]:
cols = dse.param_columns(big_oracle) + ["cost"]
big_oracle.sort_values("cost").head(5)[cols].style.format({"cost": "{:.3f}"}).hide(axis="index")

Compare with the small space: there the optimum ran 8 tasklets on 64 DPUs;
here it runs **1** tasklet on all **2048**. The right hardware parameters flip
with the problem size — one more reason they cannot be fixed before the tiling
is chosen.

## Extra · a space nobody enumerates

A matrix–matrix product on the same 32 ranks. Three loop dimensions instead of
two: nine parameters.

In [ ]:
gemm_mlir = DATA / "large" / "gemm.mlir"
gemm_space = dse.dump_space(gemm_mlir, "gemm_space")
print(dse.cinm.describe_space(gemm_space))
print(f"\nAt 60 ms per simulation, enumerating it would take "
      f"{gemm_space['feasible_size'] * 0.06 / 3600:.0f} hours on one core.")

In [ ]:
rl = dse.bo(gemm_mlir, "gemm_bo", max_evals=60, n_init=15)
pooll = rl.pool()
vl = dse.visited(pooll)

print(f"60 simulations in {rl.seconds:.1f} s")
ax = plots.best_so_far({"Bayesian, 60 evals": pooll}, n_init=15,
                       title="2.8 million feasible configurations — no oracle to compare with")
plt.show()

vl.sort_values("cost").head(3)[dse.param_columns(pooll) + ["cost"]].style.format({"cost": "{:.2f}"}).hide(axis="index")

No oracle here, and there never will be one. What the previous two spaces
bought us is *calibration*: on spaces where the answer was known, this budget
and this search landed within a few percent of it. That is the evidence one
has when applying the same procedure where the answer is not known — which is
every real case.